In [1]:
import tensorflow as tf
tf.keras.backend.clear_session()

import os
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import EfficientNetB0, preprocess_input
from tensorflow.keras import layers, models
from sklearn.utils.class_weight import compute_class_weight

In [2]:
BASE_DIR = r"C:/Users/raksh/x-ai for medical imaging/data/chest_xray_multi"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

MODEL_SAVE_PATH = "backend/saved_models/chest_multidisease_final.keras"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_PHASE1 = 15
EPOCHS_PHASE2 = 10

In [3]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=8,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_gen = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

NUM_CLASSES = train_gen.num_classes
print("Classes:", train_gen.class_indices)

Found 2824 images belonging to 7 classes.
Found 567 images belonging to 7 classes.
Found 566 images belonging to 7 classes.
Classes: {'CARDIOMEGALY': 0, 'COVID19': 1, 'EFFUSION': 2, 'NORMAL': 3, 'PNEUMONIA': 4, 'PNEUMOTHORAX': 5, 'TUBERCULOSIS': 6}


In [4]:
labels = train_gen.classes

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)

Class Weights: {0: np.float64(1.0085714285714287), 1: np.float64(1.0136396267049534), 2: np.float64(1.003553660270078), 3: np.float64(0.988795518207283), 4: np.float64(0.988795518207283), 5: np.float64(1.0085714285714287), 6: np.float64(0.988795518207283)}


In [5]:
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = layers.Input(shape=(224,224,3))
x = base_model(inputs, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)

x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.4)(x)

outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,384,426 (16.73 MB)

 Trainable params: 332,295 (1.27 MB)

 Non-trainable params: 4,052,131 (15.46 MB)

In [6]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [7]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "backend/saved_models/chest_phase1_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6
    )
]

In [8]:
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_PHASE1,
    class_weight=class_weights,
    callbacks=callbacks
)

Epoch 1/15
89/89 ━━━━━━━━━━━━━━━━━━━━ 181s 2s/step - accuracy: 0.6208 - loss: 1.1454 - val_accuracy: 0.6067 - val_loss: 0.9146 - learning_rate: 0.0010
Epoch 2/15
89/89 ━━━━━━━━━━━━━━━━━━━━ 139s 2s/step - accuracy: 0.7206 - loss: 0.7904 - val_accuracy: 0.6843 - val_loss: 0.7557 - learning_rate: 0.0010
Epoch 3/15
89/89 ━━━━━━━━━━━━━━━━━━━━ 136s 2s/step - accuracy: 0.7436 - loss: 0.6786 - val_accuracy: 0.6790 - val_loss: 0.6782 - learning_rate: 0.0010
Epoch 4/15
89/89 ━━━━━━━━━━━━━━━━━━━━ 140s 2s/step - accuracy: 0.7567 - loss: 0.6219 - val_accuracy: 0.6949 - val_loss: 0.6880 - learning_rate: 0.0010
Epoch 5/15
89/89 ━━━━━━━━━━━━━━━━━━━━ 137s 2s/step - accuracy: 0.7865 - loss: 0.5562 - val_accuracy: 0.7249 - val_loss: 0.6174 - learning_rate: 0.0010
Epoch 6/15
89/89 ━━━━━━━━━━━━━━━━━━━━ 137s 2s/step - accuracy: 0.7999 - loss: 0.5108 - val_accuracy: 0.7319 - val_loss: 0.6505 - learning_rate: 0.0010
Epoch 7/15
89/89 ━━━━━━━━━━━━━━━━━━━━ 135s 2s/step - accuracy: 0.7975 - loss: 0.5259 - val_acc

In [9]:
base_model.trainable = True

for layer in base_model.layers[:-60]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [10]:
callbacks_ft = [
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_SAVE_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6
    )
]

In [11]:
history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_PHASE2,
    class_weight=class_weights,
    callbacks=callbacks_ft
)

Epoch 1/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 204s 2s/step - accuracy: 0.6555 - loss: 0.9289 - val_accuracy: 0.6949 - val_loss: 0.7594 - learning_rate: 1.0000e-05
Epoch 2/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 170s 2s/step - accuracy: 0.7001 - loss: 0.8136 - val_accuracy: 0.6720 - val_loss: 0.8390 - learning_rate: 1.0000e-05
Epoch 3/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 169s 2s/step - accuracy: 0.7050 - loss: 0.7530 - val_accuracy: 0.6737 - val_loss: 0.8231 - learning_rate: 1.0000e-05
Epoch 4/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 171s 2s/step - accuracy: 0.7341 - loss: 0.6846 - val_accuracy: 0.6720 - val_loss: 0.8130 - learning_rate: 1.0000e-05
Epoch 5/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 163s 2s/step - accuracy: 0.7326 - loss: 0.6783 - val_accuracy: 0.6649 - val_loss: 0.8048 - learning_rate: 3.0000e-06
Epoch 6/10
89/89 ━━━━━━━━━━━━━━━━━━━━ 164s 2s/step - accuracy: 0.7369 - loss: 0.6699 - val_accuracy: 0.6720 - val_loss: 0.8008 - learning_rate: 3.0000e-06


In [12]:
test_loss, test_acc = model.evaluate(test_gen)
print("✅ Final Test Accuracy:", test_acc)

18/18 ━━━━━━━━━━━━━━━━━━━━ 21s 1s/step - accuracy: 0.7032 - loss: 0.8704
✅ Final Test Accuracy: 0.703180193901062


In [13]:
from tensorflow.keras.models import load_model

phase1_model = load_model("backend/saved_models/chest_phase1_best.keras")

In [15]:
test_loss, test_acc = phase1_model.evaluate(test_gen)

print("✅ Phase 1 Test Accuracy:", test_acc)

18/18 ━━━━━━━━━━━━━━━━━━━━ 24s 1s/step - accuracy: 0.6802 - loss: 0.8006
✅ Phase 1 Test Accuracy: 0.6802120208740234


In [16]:
FINAL_MODEL_PATH = "backend/saved_models/chest_final.keras"

phase1_model.save(FINAL_MODEL_PATH)

print("✅ Final model saved as:", FINAL_MODEL_PATH)

✅ Final model saved as: backend/saved_models/chest_final.keras


In [17]:
import numpy as np

x_batch, y_batch = next(test_gen)
preds = model.predict(x_batch)
pred_labels = np.argmax(preds, axis=1)
true_labels = np.argmax(y_batch, axis=1)

print("Predicted:", pred_labels[:20])
print("True     :", true_labels[:20])
print("Class map:", train_gen.class_indices)


1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step
Predicted: [5 0 0 5 2 2 5 2 0 5 2 2 2 0 5 0 0 0 0 2]
True     : [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Class map: {'CARDIOMEGALY': 0, 'COVID19': 1, 'EFFUSION': 2, 'NORMAL': 3, 'PNEUMONIA': 4, 'PNEUMOTHORAX': 5, 'TUBERCULOSIS': 6}
